# Solving a Quadratic Equation with `Z3`.

In this notebook we are going to solve the following exercise using the constraint solver `Z3`.
*  A train travels at a uniform speed for 360 miles.  
*  The train would have taken 48 minutes less to travel the same distance 
   if it had been faster by 5 miles per hour.

**Find the speed of the train!**

**Hints**

* As the speed is a real number you should declare this variable via the `Z3` function `Real` instead of using
  the function `Int`.
* When you formulate the information given above, you will get a system of *non-linear* equations,
  which is equivalent to a quadratic equation.  This quadratic equation has two different solutions.
  One of these solutions is negative.  In order to exclude the negative solution you need to add a
  constraint stating that the speed of the train has to be greater than zero.

The next two cells install and load `Z3`.

In [1]:
// load the formatting setup and the style sheet, install Z3 if necessary
#load "../setup.fsx"
installZ3 ()
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
// load Z3
#load "../z3.fsx"
open Microsoft.Z3

In `Z3`, every variable has to be declared.  The function `Real` declares a variable with the given name that can take real numbers as values. 
We will use two variables:
 - `speed` is the speed of the train,
 - `time` is the time that was needed to travel the distance.

In [3]:
let speed = Real "speed"
let time  = Real "time"

Next, we create a *solver* object.

In [4]:
let S = newSolver ()

If we denote the velocity of the train with $v$ and the time with $t$, we get following two equations:
$$ v * t = 360 \quad \wedge \quad (v + 5) * (t - \frac{4}{5}) = 360 $$
If we let ChatGPT 4 work this out, we get the following solution:

---

Given the system of equations:

1. $v \cdot t = 360$
2. $(v + 5) \cdot \left(t - \frac{4}{5}\right) = 360$

From the first equation, we can express $v$ in terms of $t$:
$$v = \frac{360}{t}$$
Substituting $v$ from the first equation into the second equation:
$$\left(\frac{360}{t} + 5\right) \cdot \left(t - \frac{4}{5}\right) = 360$$
Expanding the equation:
$$\frac{360}{t} \cdot t + 5 \cdot t - \frac{360 \cdot 4}{5t} - 4 = 360$$
Simplifying:
$$360 + 5t - \frac{288}{t} - 4 = 360$$
Subtract 360 from both sides:
$$5t - \frac{288}{t} - 4 = 0$$
To eliminate the fraction, multiply the entire equation by $t$:
$$5t^2 - 288 - 4t = 0$$
This gives us a quadratic equation:
$$5t^2 - 4t - 288 = 0$$
To solve this quadratic equation, we can use the quadratic formula, where $a = 5$, $b = -4$, and $c = -288$:
$$t = \frac{-b \pm \sqrt{b^2 - 4ac}}{2a}$$
Substituting the values of $a$, $b$, and $c$:
$$t = \frac{-(-4) \pm \sqrt{(-4)^2 - 4 \cdot 5 \cdot (-288)}}{2 \cdot 5}$$

$$t = \frac{4 \pm \sqrt{16 + 5760}}{10}$$

$$t = \frac{4 \pm \sqrt{5776}}{10}$$

$$t = \frac{4 \pm 76}{10}$$

So, we have two possible values for $t$:

$$t_1 = \frac{4 + 76}{10} = \frac{80}{10} = 8$$

$$t_2 = \frac{4 - 76}{10} = \frac{-72}{10} = -7.2$$

Since time cannot be negative in this context, we discard $t_2$, leaving us with $t = 8$.

Substituting $t = 8$ back into the equation $v = \frac{360}{t}$ to find $v$:
$$v = \frac{360}{8} = 45$$
Therefore, the solution to the system of equations is $v = 45$ and $t = 8$.

---

In order to prevent `Z3` from finding the negative solution, we add a constraint that the speed has to be positive.  The function `Q(4, 5)` creates the rational number $\frac{4}{5}$.

In [5]:
S.Add(speed * time .= 360)
S.Add((speed + 5) * (time - Q(4, 5)) .= 360)
S.Add(speed .> 0)

The method `Check` examines whether the given set of constraints is satisfiable.
In general, it can return one of the following results:
- `sat`   is returned if the problem is solvable, (`sat` is short for *satisfiable*)
- `unsat` is returned if the problem is unsatisfiable,
- `unknown` is returned if the constraint solver is not powerful enough to solve the given problem.

In [6]:
S.Check()

sat

To extract the solution of the given problem we use the property `Model`.

In [7]:
let solution = S.Model
solution

[speed = 45, time = 8]

The method `Evaluate` of a model returns the value of an expression in this model.

In [8]:
let s = solution.Evaluate(speed, true)

In [9]:
printfn "The speed of the train is %s miles per hour." (z3ToString s)

The speed of the train is 

45

 miles per hour.